<div style="background:linear-gradient(135deg,#06080d,#0e1623,#06080d);border-radius:14px;padding:36px 32px;text-align:center;margin-bottom:8px;border:1px solid #1a2640">
  <h1 style="font-family:monospace;color:#00e5aa;font-size:2.2em;letter-spacing:4px;margin:0">◈ UNIVERSR</h1>
  <p style="color:#3a5270;font-family:monospace;font-size:0.85em;margin:10px 0 0;letter-spacing:1px">// vocoder-free audio super-resolution · flow matching · 8 → 48 kHz</p>
  <div style="margin-top:16px;display:flex;justify-content:center;gap:10px;flex-wrap:wrap">
    <span style="background:#00e5aa0a;border:1px solid #00e5aa33;border-radius:3px;padding:3px 12px;font-family:monospace;font-size:0.75em;color:#00b888">Speech · General</span>
    <span style="background:#00e5aa0a;border:1px solid #00e5aa33;border-radius:3px;padding:3px 12px;font-family:monospace;font-size:0.75em;color:#00b888">HuggingFace ✓</span>
    <span style="background:#00e5aa0a;border:1px solid #00e5aa33;border-radius:3px;padding:3px 12px;font-family:monospace;font-size:0.75em;color:#00b888">Gradio GUI</span>
    <span style="background:#00e5aa0a;border:1px solid #00e5aa33;border-radius:3px;padding:3px 12px;font-family:monospace;font-size:0.75em;color:#00b888">iSTFT Waveform</span>
  </div>
</div>

In [ ]:
#@title ① Download app.py — Download the GUI file

import base64, os, subprocess, sys

# Install gdown (pre-installed on Colab, just in case)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
import gdown

# Encoded Google Drive file ID
_e = 'MXJzamtIRlZLTTRYN190dEtDdERtSnNQUk1OUmV1UVpi'
_id = base64.b64decode(_e).decode()

dest = 'universr_app.py'
if not os.path.exists(dest):
    print('Downloading universr_app.py ...')
    gdown.download(id=_id, output=dest, quiet=False)
    print(f'✅ Saved: {dest}  ({os.path.getsize(dest)//1024} KB)')
else:
    print(f'✅ Already exists: {dest}')

In [ ]:
#@title ② Set up dependencies — required only on first run

!pip install -q gradio torch torchaudio einops omegaconf librosa matplotlib soundfile requests tqdm torchdiffeq pillow
print("✅ All dependencies installed.")

import torch, torchaudio, gradio, librosa
print(f"torch      {torch.__version__}")
print(f"torchaudio {torchaudio.__version__}")
print(f"gradio     {gradio.__version__}")
print(f"librosa    {librosa.__version__}")
print(f"CUDA       {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '— CPU only'}")

In [ ]:
#@title ③ Launch GUI — public link is created automatically, weights are downloaded on first model selection (~230 MB)

import threading, time, sys, gradio as gr

# ── 1. Close old instances ────────────────────────────────────────────────────
try:
    gr.close_all()
    time.sleep(1)
except Exception:
    pass

# ── 2. Import + reload app.py ─────────────────────────────────────────────────
import importlib, universr_app as _app
importlib.reload(_app)
demo = _app.build_ui()

# ── 3. Launch in background (don't touch stdout — uvicorn will crash) ─────────
def _run():
    demo.launch(
        share       = True,
        server_name = "0.0.0.0",
        server_port = 7860,
        inline      = False,
        quiet       = True,
        debug       = False,
        prevent_thread_lock = True,
    )

t = threading.Thread(target=_run, daemon=True)
t.start()

# ── 4. Wait until URL is ready (max 20 sec) ───────────────────────────────────
url = None
for _ in range(20):
    time.sleep(1)
    try:
        if demo.share_url:
            url = demo.share_url
            break
    except Exception:
        pass

if not url:
    url = "http://localhost:7860"

# ── 5. Single clean link only — Gradio's own print is silenced via quiet=True ─
import re
from IPython.display import display, HTML, clear_output
clear_output(wait=True)   # Clear the "Running on public URL" line

display(HTML(f"""
<div style="font-family:'IBM Plex Mono',monospace;background:#06080d;
            border:1px solid #00e5aa55;border-radius:8px;
            padding:20px 28px;margin:12px 0;display:inline-block;
            box-shadow:0 0 20px #00e5aa11">
  <div style="color:#2a5a78;font-size:0.78em;margin-bottom:10px">// UniverSR GUI · active</div>
  <a href="{url}" target="_blank"
     style="color:#00e5aa;font-size:1.15em;font-weight:700;
            text-decoration:none;letter-spacing:1px">⚡ &nbsp;{url}</a>
  <div style="color:#2a5a78;font-size:0.75em;margin-top:10px">
    Click to open in a new tab &nbsp;·&nbsp; valid for 1 week
  </div>
</div>
"""))

<div style="text-align:center;margin-top:24px;padding:14px;border-top:1px solid #1a2640;font-family:monospace;font-size:0.78em;color:#3a5270">
  Woongjib Choi et al. &nbsp;·&nbsp;
  <a href="https://arxiv.org/abs/2510.00771" style="color:#2a5a78">arXiv:2510.00771</a> &nbsp;·&nbsp;
  <a href="https://github.com/woongzip1/UniverSR" style="color:#2a5a78">GitHub</a> &nbsp;·&nbsp;
  <a href="https://huggingface.co/woongzip1/universr-audio" style="color:#2a5a78">HF General</a> &nbsp;·&nbsp;
  <a href="https://huggingface.co/woongzip1/universr-speech" style="color:#2a5a78">HF Speech</a>
</div>